In [1]:
import numpy as np
import pandas as pd
import warnings
import altair as alt
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier

warnings.filterwarnings("ignore")
%matplotlib inline
pd.options.display.precision = 15
alt.renderers.enable('mimetype')

%env JOBLIB_TEMP_FOLDER=/tmp

env: JOBLIB_TEMP_FOLDER=/tmp


In [2]:
from pathlib import Path
import pandas as pd

# Detect environment
try:
    import google.colab
    IS_COLAB = True
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    IS_COLAB = False

# Root paths
if IS_COLAB:
    ROOT = Path("/content/drive/MyDrive/minor-thesis")
else:
    ROOT = Path.cwd()

DATASET_PATH = ROOT / "dataset"
SAVED_PATH = ROOT / "saved"

print(f"Running on {'Google Colab' if IS_COLAB else 'Local'}")
print(f"Dataset path: {DATASET_PATH}")


Running on Local
Dataset path: d:\source\RMIT\master-of-ai-new\2026-semester-02\minor-thesis\dataset


In [3]:
train_identity = pd.read_csv(f'{DATASET_PATH}/train_identity.csv')
train_transaction = pd.read_csv(f'{DATASET_PATH}/train_transaction.csv')
test_identity = pd.read_csv(f'{DATASET_PATH}/test_identity.csv')
test_transaction = pd.read_csv(f'{DATASET_PATH}/test_transaction.csv')

# let's combine the data and work with the whole dataset
train = pd.merge(train_transaction, train_identity, on='TransactionID', how='left')
test = pd.merge(test_transaction, test_identity, on='TransactionID', how='left')

RANDOM_SEED = 42
START_DATE = "2026-01-01"

In [4]:
del train_identity, train_transaction, test_identity, test_transaction

In [5]:
print(train.columns.tolist())

['TransactionID', 'isFraud', 'TransactionDT', 'TransactionAmt', 'ProductCD', 'card1', 'card2', 'card3', 'card4', 'card5', 'card6', 'addr1', 'addr2', 'dist1', 'dist2', 'P_emaildomain', 'R_emaildomain', 'C1', 'C2', 'C3', 'C4', 'C5', 'C6', 'C7', 'C8', 'C9', 'C10', 'C11', 'C12', 'C13', 'C14', 'D1', 'D2', 'D3', 'D4', 'D5', 'D6', 'D7', 'D8', 'D9', 'D10', 'D11', 'D12', 'D13', 'D14', 'D15', 'M1', 'M2', 'M3', 'M4', 'M5', 'M6', 'M7', 'M8', 'M9', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'V29', 'V30', 'V31', 'V32', 'V33', 'V34', 'V35', 'V36', 'V37', 'V38', 'V39', 'V40', 'V41', 'V42', 'V43', 'V44', 'V45', 'V46', 'V47', 'V48', 'V49', 'V50', 'V51', 'V52', 'V53', 'V54', 'V55', 'V56', 'V57', 'V58', 'V59', 'V60', 'V61', 'V62', 'V63', 'V64', 'V65', 'V66', 'V67', 'V68', 'V69', 'V70', 'V71', 'V72', 'V73', 'V74', 'V75', 'V76', 'V77', 'V78', 'V79', 'V80', 'V81', 'V

In [6]:
train['isFraud'].value_counts(normalize=True)

isFraud
0    0.965009990855827
1    0.034990009144173
Name: proportion, dtype: float64

In [7]:
# columns grouping
# transaction columns
transaction_cols = ["TransactionID", "TransactionDT", "TransactionAmt"]
# product columns
product_cols = ["ProductCD"]
# card columns
card_cols = ["card1", "card2", "card3", "card4", "card5", "card6"]
# address columns
address_cols = ["addr1", "addr2"]
# email columns
email_cols = ["P_emaildomain", "R_emaildomain"]
# distance columns
distance_cols = ["dist1", "dist2"]
# matching features columns
matching_cols = ["M1", "M2", "M3", "M4", "M5", "M6", "M7", "M8", "M9"]
# count columns
count_cols = [f"C{i}" for i in range(1, 15)]
# delay columns
delay_cols = [f"D{i}" for i in range(1, 16)]
# v columns
v_cols = [f"V{i}" for i in range(1, 340)]
# device columns
device_cols = ["DeviceType", "DeviceInfo"]
# identity columns
identity_cols = [f"id_{str(i).zfill(2)}" for i in range(1, 39)]
# device metadata columns
# id_30: Operating System,
# id_31: Browser
# id_32: screen color depth
# id_33: screen resolution
device_metadata_cols = ["DeviceType", "DeviceInfo", "id_30", "id_31", "id_32", "id_33"]

In [8]:
check = train[
    [
        "TransactionID",
        "TransactionDT",
        "TransactionAmt",
        "DeviceType",
        "DeviceInfo",
        "id_30",
        "id_31",
        "id_32",
        "id_33",
        "isFraud",
    ]
]
check[check["isFraud"] == True]

,TransactionID,TransactionDT,TransactionAmt,DeviceType,DeviceInfo,id_30,id_31,id_32,id_33,isFraud
203,2987203,89760,445.000000000000000,NaN,NaN,NaN,NaN,NaN,NaN,1
240,2987240,90193,37.097999999999999,mobile,Redmi Note 4 Build/MMB29M,NaN,chrome 54.0 for android,NaN,NaN,1
243,2987243,90246,37.097999999999999,mobile,Redmi Note 4 Build/MMB29M,NaN,chrome 54.0 for android,NaN,NaN,1
245,2987245,90295,37.097999999999999,mobile,Redmi Note 4 Build/MMB29M,NaN,chrome 54.0 for android,NaN,NaN,1
288,2987288,90986,155.520999999999987,mobile,NaN,NaN,chrome 62.0 for ios,NaN,NaN,1
...,...,...,...,...,...,...,...,...,...,...
590361,3577361,15807368,1224.000000000000000,NaN,NaN,NaN,NaN,NaN,NaN,1
590364,3577364,15807516,69.963999999999999,mobile,SAMSUNG SM-J700M Build/MMB29K,NaN,samsung browser 6.4,NaN,NaN,1
590368,3577368,15807677,100.000000000000000,mobile,iOS Device,iOS 11.3.0,mobile safari 11.0,32.0,2208x1242,1
590372,3577372,15807758,117.000000000000000,NaN,NaN,NaN,NaN,NaN,NaN,1


In [9]:
# Date feature engineering with TransactionDT
train["DT"] = pd.to_datetime(START_DATE) + pd.to_timedelta(
    train["TransactionDT"], unit="s"
)

# Extract temporal features
train["DT_month"] = train["DT"].dt.month
train["DT_week"] = train["DT"].dt.isocalendar().week.astype("int32")
train["DT_day"] = train["DT"].dt.day
train["DT_weekday"] = train["DT"].dt.weekday
train["DT_hour"] = train["DT"].dt.hour

# Optional: remove the temporary datetime column
train.drop(columns=["DT"], inplace=True)

In [10]:
# Construct UID
train["uid"] = (
    train["card1"].astype(str) + "_" +
    train["card2"].astype(str) + "_" +
    train["card3"].astype(str) + "_" +
    train["card5"].astype(str)
)

# Construct UID2
train["uid2"] = (
    train["uid"] + "_" +
    train["addr1"].astype(str) + "_" +
    train["P_emaildomain"].astype(str)
)

## Drop columns that are ≥90% empty

Missingness is measured **before** fill/encode, on the raw merged train table. A value counts as empty if it is null/NaN or a blank / `"nan"` / `"none"` / `"null"` string.

Columns at or above 90% empty are dropped from a **copy** of the pipeline. `merged_train.parquet` stays the full filled table; `merged_train_reduced.parquet` is the same rows with those columns removed.

`isFraud`, `TransactionID`, `TransactionDT`, `TransactionAmt`, `uid`, and `uid2` are never dropped.

In [11]:
EMPTY_THRESHOLD = 0.90
ALWAYS_KEEP = {
    "isFraud",
    "TransactionID",
    "TransactionDT",
    "TransactionAmt",
    "uid",
    "uid2",
}


def empty_fraction(series: pd.Series) -> float:
    """Share of null / blank / sentinel-empty values in a column."""
    if pd.api.types.is_object_dtype(series) or pd.api.types.is_string_dtype(series):
        text = series.astype("string")
        stripped = text.str.strip()
        empty = (
            series.isna()
            | stripped.eq("")
            | stripped.str.lower().isin(["nan", "none", "null", "<na>"])
        )
        return float(empty.mean())
    return float(series.isna().mean())


empty_rates = (
    pd.Series({col: empty_fraction(train[col]) for col in train.columns}, name="empty_frac")
    .sort_values(ascending=False)
    .to_frame()
)
empty_rates["drop"] = (empty_rates["empty_frac"] >= EMPTY_THRESHOLD) & (
    ~empty_rates.index.isin(ALWAYS_KEEP)
)
cols_drop_90 = empty_rates.index[empty_rates["drop"]].tolist()

print(f"Train shape before drop: {train.shape}")
print(f"Columns with empty_frac >= {EMPTY_THRESHOLD:.0%}: {len(cols_drop_90)}")
print(cols_drop_90)
display(empty_rates.head(20))

SAVED_PATH.mkdir(parents=True, exist_ok=True)
empty_rates.to_csv(SAVED_PATH / "column_empty_rates.csv")
pd.Series(cols_drop_90, name="column").to_json(
    SAVED_PATH / "cols_drop_90.json", orient="values"
)
print(f"Wrote {SAVED_PATH / 'column_empty_rates.csv'}")
print(f"Wrote {SAVED_PATH / 'cols_drop_90.json'}")

Train shape before drop: (590540, 441)
Columns with empty_frac >= 90%: 12
['id_24', 'id_25', 'id_08', 'id_07', 'id_21', 'id_26', 'id_22', 'id_27', 'id_23', 'dist2', 'D7', 'id_18']


,empty_frac,drop
id_24,0.991961594472855,True
id_25,0.991309648796017,True
id_08,0.991270701391946,True
id_07,0.991270701391946,True
id_21,0.991263927930369,True
id_26,0.991257154468791,True
id_22,0.991246994276425,True
id_27,0.991246994276425,True
id_23,0.991246994276425,True
dist2,0.936283740305483,True


Wrote d:\source\RMIT\master-of-ai-new\2026-semester-02\minor-thesis\saved\column_empty_rates.csv
Wrote d:\source\RMIT\master-of-ai-new\2026-semester-02\minor-thesis\saved\cols_drop_90.json


In [12]:
# reduce memory
def reduce_mem_usage(df, verbose=True):
    """
    Iterate through all columns of a dataframe and
    modify the data type to reduce memory usage.
    """
    print("reduce memory")

    start_mem = df.memory_usage(deep=True).sum() / 1024**2

    for col in df.columns:
        col_type = df[col].dtype

        # Skip datetime columns
        if pd.api.types.is_datetime64_any_dtype(df[col]):
            continue

        # Integer columns
        if pd.api.types.is_integer_dtype(col_type):
            c_min = df[col].min()
            c_max = df[col].max()

            if c_min >= np.iinfo(np.int8).min and c_max <= np.iinfo(np.int8).max:
                df[col] = df[col].astype(np.int8)
            elif c_min >= np.iinfo(np.int16).min and c_max <= np.iinfo(np.int16).max:
                df[col] = df[col].astype(np.int16)
            elif c_min >= np.iinfo(np.int32).min and c_max <= np.iinfo(np.int32).max:
                df[col] = df[col].astype(np.int32)
            else:
                df[col] = df[col].astype(np.int64)

        # Float columns
        elif pd.api.types.is_float_dtype(col_type):
            df[col] = df[col].astype(np.float32)

        # Object columns
        elif col_type == object:
            continue

    end_mem = df.memory_usage(deep=True).sum() / 1024**2

    if verbose:
        print(f"Memory usage before: {start_mem:.2f} MB")
        print(f"Memory usage after : {end_mem:.2f} MB")
        print(f"Decreased by {(100 * (start_mem - end_mem) / start_mem):.1f}%")

    return df


def encode_categorical_columns(df):
    print("encode categorical columns")

    # Handle M1-M3 and M5-M9
    matching_binary_cols = ["M1", "M2", "M3", "M5", "M6", "M7", "M8", "M9"]

    for col in matching_binary_cols:
        if col in df.columns:
          df[col] = df[col].map({"T": 1, "F": 0}).fillna(-1).astype("int8")

    # Handle M4 separately
    if "M4" in df.columns:
        df["M4"] = LabelEncoder().fit_transform(df["M4"].fillna("Missing").astype(str))


    # Label encode remaining
    # categorical columns
    categorical_cols = df.select_dtypes(include=["object", "category"]).columns

    for col in categorical_cols:

        # M4 has already been encoded
        if col == "M4":
            continue

        df[col] = LabelEncoder().fit_transform(df[col].fillna("Missing").astype(str))

    return df


# Handle missing values
def handle_missing_values(df):
    print("handle missing values")
    # Numerical columns
    num_cols = df.select_dtypes(include=["int", "float"]).columns

    # Fill missing numerical values with -999
    df[num_cols] = df[num_cols].fillna(-999)

    # Object / categorical columns
    cat_cols = df.select_dtypes(include=["object", "category"]).columns

    # Preserve missing information as string
    df[cat_cols] = df[cat_cols].fillna("Missing")

    return df

In [13]:
train = encode_categorical_columns(train)
train = handle_missing_values(train)
train = reduce_mem_usage(train)

encode categorical columns
handle missing values
reduce memory
Memory usage before: 1944.11 MB
Memory usage after : 930.38 MB
Decreased by 52.1%


In [14]:
# save the data
train.to_parquet(f"{DATASET_PATH}/merged_train.parquet", index=False)
print(f"saved train data  {train.shape}")

drop_in_train = [c for c in cols_drop_90 if c in train.columns]
train_reduced = train.drop(columns=drop_in_train)
train_reduced.to_parquet(f"{DATASET_PATH}/merged_train_reduced.parquet", index=False)
print(f"saved reduced train data  {train_reduced.shape}  dropped {len(drop_in_train)} columns")

saved train data  (590540, 441)
saved reduced train data  (590540, 429)  dropped 12 columns


## Competition test parquet

`merged_test.parquet` must come from the IEEE **test** CSVs (no `isFraud`), with encodings aligned to `merged_train.parquet`. This cell rebuilds it if the file is missing or is still a copy of train.


In [15]:
"""Rebuild dataset/merged_test.parquet from the IEEE test CSVs.

01_clean_dataset.ipynb accidentally wrote merged_train twice, so the current
merged_test.parquet is an identical copy of train (including isFraud).
The competition test file has no labels. Encoders are aligned to the existing
merged_train.parquet so RF / LightGBM / XGBoost / GNN features match.
"""

import gc
from pathlib import Path

import numpy as np
import pandas as pd

START_DATE = "2026-01-01"
MATCHING_BINARY = ["M1", "M2", "M3", "M5", "M6", "M7", "M8", "M9"]


def add_time_and_uid(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    dt = pd.to_datetime(START_DATE) + pd.to_timedelta(df["TransactionDT"], unit="s")
    df["DT_month"] = dt.dt.month
    df["DT_week"] = dt.dt.isocalendar().week.astype("int32")
    df["DT_day"] = dt.dt.day
    df["DT_weekday"] = dt.dt.weekday
    df["DT_hour"] = dt.dt.hour
    df["uid"] = (
        df["card1"].astype(str)
        + "_"
        + df["card2"].astype(str)
        + "_"
        + df["card3"].astype(str)
        + "_"
        + df["card5"].astype(str)
    )
    df["uid2"] = (
        df["uid"]
        + "_"
        + df["addr1"].astype(str)
        + "_"
        + df["P_emaildomain"].astype(str)
    )
    return df


def load_raw(split: str) -> pd.DataFrame:
    trans = pd.read_csv(DATASET_PATH / f"{split}_transaction.csv")
    ident = pd.read_csv(DATASET_PATH / f"{split}_identity.csv")
    ident.columns = [c.replace("-", "_") for c in ident.columns]
    df = trans.merge(ident, on="TransactionID", how="left")
    del trans, ident
    return add_time_and_uid(df)


def compact_feature_cols(columns) -> list[str]:
    cols = [
        c
        for c in columns
        if c not in {"isFraud", "TransactionID", "uid", "uid2"}
        and not str(c).startswith("id")
        and not str(c).startswith("V")
    ]
    for extra in ("uid", "uid2"):
        if extra in columns and extra not in cols:
            cols.append(extra)
    return cols


def test_is_train_copy(train: pd.DataFrame, test: pd.DataFrame) -> bool:
    if train.shape != test.shape:
        return False
    if "TransactionID" not in test.columns:
        return False
    return set(train["TransactionID"]) == set(test["TransactionID"])


def encode_like_train(raw_train: pd.DataFrame, encoded_train: pd.DataFrame, raw_test: pd.DataFrame) -> pd.DataFrame:
    ordered = [c for c in encoded_train.columns if c != "isFraud"]
    raw_train = raw_train.set_index("TransactionID")
    encoded_train = encoded_train.set_index("TransactionID")
    raw_test = raw_test.set_index("TransactionID")
    n = len(raw_test)
    data = {}

    for col in ordered:
        if col == "TransactionID":
            data[col] = raw_test.index.to_numpy()
            continue
        if col not in raw_test.columns:
            data[col] = np.full(n, -999, dtype=np.float32)
            continue

        if col in MATCHING_BINARY:
            data[col] = raw_test[col].map({"T": 1, "F": 0}).fillna(-1).to_numpy(dtype=np.int8)
            continue

        src = raw_train[col] if col in raw_train.columns else None
        is_cat = src is not None and (
            col in {"uid", "uid2"}
            or pd.api.types.is_object_dtype(src)
            or pd.api.types.is_string_dtype(src)
            or str(src.dtype) in {"category", "string"}
        )
        if is_cat:
            keys = src.fillna("Missing").astype(str)
            mapping = (
                pd.DataFrame({"k": keys.to_numpy(), "v": encoded_train[col].to_numpy()})
                .drop_duplicates("k")
                .set_index("k")["v"]
            )
            mapped = raw_test[col].fillna("Missing").astype(str).map(mapping)
            fill = int(encoded_train[col].max()) + 1 if encoded_train[col].notna().any() else -1
            data[col] = pd.to_numeric(mapped, errors="coerce").fillna(fill).to_numpy()
            continue

        data[col] = pd.to_numeric(raw_test[col], errors="coerce").fillna(-999).to_numpy()

    return pd.DataFrame(data)[ordered]


def reduce_like_train(df: pd.DataFrame, encoded_train: pd.DataFrame) -> pd.DataFrame:
    for col in df.columns:
        if col not in encoded_train.columns:
            continue
        dt = encoded_train[col].dtype
        arr = pd.to_numeric(df[col], errors="coerce").to_numpy(dtype=np.float64, copy=False)
        fill = -1.0 if pd.api.types.is_integer_dtype(dt) else -999.0
        arr = np.nan_to_num(arr, nan=fill, posinf=fill, neginf=fill)
        if pd.api.types.is_integer_dtype(dt):
            info = np.iinfo(dt)
            arr = np.clip(arr, info.min, info.max)
        df[col] = arr.astype(dt, copy=False)
    return df


def ensure_merged_test(force: bool = False) -> Path:
    train_path = DATASET_PATH / "merged_train.parquet"
    test_path = DATASET_PATH / "merged_test.parquet"
    encoded_train = pd.read_parquet(train_path)

    if test_path.exists() and not force:
        test = pd.read_parquet(test_path)
        if not test_is_train_copy(encoded_train, test) and "isFraud" not in test.columns:
            print(f"merged_test.parquet already looks like the competition test: {test.shape}")
            return test_path
        if test_is_train_copy(encoded_train, test):
            print(
                "WARNING: merged_test.parquet is an identical copy of merged_train "
                "(01_clean_dataset.ipynb saved train twice). "
                "Labeled test metrics on this file mix train and holdout rows. "
                "Re-run the test-parquet cell with force=True."
            )
            return test_path
        print(f"Using existing merged_test.parquet: {test.shape}")
        return test_path

    print("Loading raw train/test CSVs...")
    raw_train = load_raw("train")
    raw_test = load_raw("test")
    print(f"raw train {raw_train.shape}  raw test {raw_test.shape}")

    encoded = encode_like_train(raw_train, encoded_train, raw_test)
    del raw_train, raw_test
    gc.collect()
    encoded = reduce_like_train(encoded, encoded_train)
    encoded.to_parquet(test_path, index=False)
    print(f"Wrote {test_path}  shape={encoded.shape}  columns={len(encoded.columns)}")
    print("Test has no isFraud labels (IEEE-CIS competition test).")
    return test_path

ensure_merged_test(force=False)


merged_test.parquet already looks like the competition test: (506691, 440)


WindowsPath('d:/source/RMIT/master-of-ai-new/2026-semester-02/minor-thesis/dataset/merged_test.parquet')